# 🔬 超声视频 VLM 过滤

In [1]:
import sys, os, glob
os.chdir('..')
sys.path.append('src')
from ablation_input_modes import *

# 找所有视频
videos = glob.glob("UltrasoundCrawler_KeyCode_20260323_v2/output/20260520_162816_youtube/media/**/*.mp4", recursive=True)
print(f"Found {len(videos)} videos")

# 加载模型
model, processor, device = load_model()

# 批量分类
results = []
for i, v in enumerate(videos):
    print(f"\n[{i+1}/{len(videos)}] {Path(v).stem}")
    try:
        r = run_mode3(model, processor, device, v, num_frames=8)
        r["video_path"] = v
        r["video_id"] = Path(v).stem
        results.append(r)
    except Exception as e:
        print(f"  ERROR: {e}")

# 保存
import json
with open("results/video_classification.json", "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, default=str)


Found 20 videos


/Users/I761836/Desktop/Semester 3/live-ultrasound-video-understanding/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading Qwen/Qwen3-VL-2B-Instruct on mps...


Loading weights: 100%|██████████| 625/625 [00:00<00:00, 8003.59it/s]


Model loaded!

[1/20] v2uBpsEKte8

MODE 3: Video Mode (native, 1 call)


qwen-vl-utils using torchcodec to read video.
objc[13146]: Class AVFFrameReceiver is implemented in both /Users/I761836/Desktop/Semester 3/live-ultrasound-video-understanding/.venv/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x10adfc3a8) and /opt/homebrew/Cellar/ffmpeg/8.1_1/lib/libavdevice.62.3.100.dylib (0x123ab8328). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[13146]: Class AVFAudioReceiver is implemented in both /Users/I761836/Desktop/Semester 3/live-ultrasound-video-understanding/.venv/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x10adfc3f8) and /opt/homebrew/Cellar/ffmpeg/8.1_1/lib/libavdevice.62.3.100.dylib (0x123ab8378). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
[transformers] Asked to sample `fps` frames per second but no video metadata was provided which is required when sampling 

  Time: 11.1s | Tokens: 571in+97out
  Output: ```json
{
  "video_type": "pure_ultrasound",
  "has_probe_technique": false,
  "has_instructor_face": false,
  "has_slides": false,
  "anatomy_regions": [],
  "description": "A screen capture of an ultrasound machine displaying a vascular image, with the user adjusting settings and viewing the image in real-time.",
  "training_value": "high",
  "recommendation": "keep"
}
```
  Parsed: {
  "video_type": "pure_ultrasound",
  "has_probe_technique": false,
  "has_instructor_face": false,
  "has_slides": false,
  "anatomy_regions": [],
  "description": "A screen capture of an ultrasound machine displaying a vascular image, with the user adjusting settings and viewing the image in real-time.",
  "training_value": "high",
  "recommendation": "keep"
}

[2/20] AxuGkz78pBA

MODE 3: Video Mode (native, 1 call)
  Time: 6.9s | Tokens: 553in+114out
  Output: ```json
{
  "video_type": "ppt_lecture",
  "has_probe_technique": true,
  "has_instructor_face": 

In [4]:
from collections import Counter

types = [r["results"]["video_type"] for r in results if "results" in r and isinstance(r["results"], dict)]
print("Video Type Distribution:")
for vtype, count in Counter(types).most_common():
    print(f"  {vtype}: {count}")

# 按 recommendation 分组
recs = [r["results"]["recommendation"] for r in results if "results" in r and isinstance(r["results"], dict)]
print(f"\nRecommendation: {Counter(recs)}")

keep_videos = [r for r in results if isinstance(r.get("results"), dict) and r["results"].get("recommendation") == "keep"]
print(f"\nKeep ({len(keep_videos)} videos):")
for r in keep_videos:
    print(f"  {r['video_id']}: {r['results']['video_type']} - {r['results']['description']}")
    print()


Video Type Distribution:
  pure_ultrasound: 8
  mixed: 8
  hands_on_tutorial: 2
  ppt_lecture: 1
  diagram_animation: 1

Recommendation: Counter({'keep': 20})

Keep (20 videos):
  v2uBpsEKte8: pure_ultrasound - A screen capture of an ultrasound machine displaying a vascular image, with the user adjusting settings and viewing the image in real-time.

  AxuGkz78pBA: ppt_lecture - A presentation on how to use an ultrasound probe, focusing on probe movements such as compression, sweep, slide, pivot, roll, pitch, and yaw. The video includes diagrams and animations to explain the techniques.

  GhkNXh0m5Nk: hands_on_tutorial - A tutorial video demonstrating ultrasound probe movements, including sliding, tilting/fanning, rotating, rocking, and compression, with a focus on the technique and visual feedback.

  TlckvYhqaFE: pure_ultrasound - A medical ultrasound examination of the abdomen, showing the ultrasound screen and the probe being used on the patient's abdomen.

  ga3t4LCZ6P0: mixed - A